In [ ]:
# Vulkan for SAPIEN (from ManiSkill's quickstart notebook)
!mkdir -p /usr/share/vulkan/icd.d
!wget -q https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/nvidia_icd.json
!wget -q https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/10_nvidia.json
!mv nvidia_icd.json /usr/share/vulkan/icd.d
!mv 10_nvidia.json /usr/share/glvnd/egl_vendor.d/10_nvidia.json
!apt-get install -y --no-install-recommends libvulkan-dev > /dev/null

In [ ]:
!python --version
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/so101-runs'
!git clone -q -b phase0-1 https://github.com/HipsterBrown/so101-policy-learning.git /content/so101 || git -C /content/so101 pull -q
%cd /content/so101
# mplib (ManiSkill's motion-planning examples only; unused here) has no Python 3.13 wheel, so drop it via a uv override.
!pip install -q uv
!echo "mplib ; sys_platform == 'never'" > /tmp/overrides.txt
!uv pip install --system -q -e . --override /tmp/overrides.txt
import site; site.main()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUT}

In [ ]:
SEED = 1
!python scripts/ppo.py --env_id SO101Reach-v1 --seed {SEED} --exp-name reach-baseline-s{SEED} \
  --num_envs 2048 --update_epochs 8 --num_minibatches 32 --num_steps 20 --total_timesteps 2_000_000 \
  --eval_freq 3 --num_eval_envs 100 --num_eval_steps 50 \
  --output-dir {OUT} --variant baseline --results-csv {OUT}/results.csv --no-capture-video